In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
df=spark.read.format('csv')\
            .option("inferSchema", True)\
            .option('header', True)\
            .load(f"file:///home/talentum/test-jupyter/DAWPPS/json-usecases/labs/data/flim.csv")
df.show()
df.printSchema()

+---+--------------------+
| id|               movie|
+---+--------------------+
|  1|{ 'Title': 'Avata...|
|  2|{ 'Title': 'I Am ...|
|  3|{ 'Title': '300',...|
|  4|{ 'Title': 'The A...|
|  5|{ 'Title': 'The W...|
|  6|{ 'Title': 'Inter...|
+---+--------------------+

root
 |-- id: integer (nullable = true)
 |-- movie: string (nullable = true)



In [4]:
import pyspark.sql.types as T
import pyspark.sql.functions as F

# create the schema to read the json text field from flim 
# dataframe using struct type and struct field
schema = T.StructType(
    [
        T.StructField('Title', T.StringType(), True),
        T.StructField('Year', T.StringType(), True),
        T.StructField('Rated', T.StringType(), True),
        T.StructField('Released', T.StringType(), True),
        T.StructField('Genre', T.StringType(), True)
    ]
)

In [6]:
# Using the withColumn function 
# which transforms string json field (movie)
# to data structure which is defined into schema variable
movie_df_mapped = df.withColumn("movie", F.from_json("movie", schema))
movie_df_mapped.show(truncate=False)
movie_df_mapped.printSchema()


+---+-------------------------------------------------------------------------------+
|id |movie                                                                          |
+---+-------------------------------------------------------------------------------+
|1  |[Avatar, 2009, PG-13, 18 Dec 2009, ["Action","Adventure","Fantasy"]]           |
|2  |[I Am Legend, 2007, PG-13, 14 Dec 2007, ["Drama","Horror","Sci-Fi"]]           |
|3  |[300, 2006, R, 09 Mar 2007, ["Action","Drama","Fantasy"]]                      |
|4  |[The Avengers, 2012, PG-13, 04 May 2012, ["Action","Sci-Fi","Thriller"]]       |
|5  |[The Wolf of Wall Street, 2013, R, 25 Dec 2013, ["Biography","Comedy","Crime"]]|
|6  |[Interstellar, 2014, PG-13, 07 Nov 2014, ["Adventure","Drama","Sci-Fi"]]       |
+---+-------------------------------------------------------------------------------+

root
 |-- id: integer (nullable = true)
 |-- movie: struct (nullable = true)
 |    |-- Title: string (nullable = true)
 |    |-- Year: stri

In [8]:
# Using the select to get the specific columns to build 
# new Data frame which will contains following below field
movie_df = movie_df_mapped.select(F.col('id'), \
                                  F.col("movie.Title"),\
                                  F.col("movie.Year"),\
                                  F.col("movie.Rated"), \
                                  F.col("movie.Released"),\
                                  F.col("movie.Genre"))
movie_df.show(truncate=False)

+---+-----------------------+----+-----+-----------+--------------------------------+
|id |Title                  |Year|Rated|Released   |Genre                           |
+---+-----------------------+----+-----+-----------+--------------------------------+
|1  |Avatar                 |2009|PG-13|18 Dec 2009|["Action","Adventure","Fantasy"]|
|2  |I Am Legend            |2007|PG-13|14 Dec 2007|["Drama","Horror","Sci-Fi"]     |
|3  |300                    |2006|R    |09 Mar 2007|["Action","Drama","Fantasy"]    |
|4  |The Avengers           |2012|PG-13|04 May 2012|["Action","Sci-Fi","Thriller"]  |
|5  |The Wolf of Wall Street|2013|R    |25 Dec 2013|["Biography","Comedy","Crime"]  |
|6  |Interstellar           |2014|PG-13|07 Nov 2014|["Adventure","Drama","Sci-Fi"]  |
+---+-----------------------+----+-----+-----------+--------------------------------+

